# OncoFound / Pancancer 3D DINO — Inference DemoThis notebook has two parts:1. **Part 1** builds the pretraining backbone and shows its output dimensions on dummy data.2. **Part 2** loads a released feature bag and a downstream classification checkpoint, then runs forward to the classification head and prints logits.No split logic and no evaluation metrics are involved. Run from the repository root.

## Part 1 — Pretraining backbone output dimensions (dummy data)Build the DINOv3 teacher backbone from the pretrain config and inspect its embedding dimension.

In [ ]:
import torchfrom omegaconf import OmegaConffrom dinov3.models import build_model_from_cfgcfg = OmegaConf.load("configs/pretrain.yaml")model, embed_dim = build_model_from_cfg(cfg, only_teacher=True)print(f"architecture : {cfg.student.arch}")print(f"patch_size   : {cfg.student.patch_size}")print(f"in_chans     : {cfg.student.in_chans}")print(f"embed_dim    : {embed_dim}")print(f"model class  : {type(model).__name__}")

The backbone is built on a *meta* device (no parameters allocated). To run a real forward pass we materialize it on CPU and fill it with random weights — this is only to demonstrate the output shapes, no pretrained weights are required.

In [ ]:
model = model.to_empty(device="cpu")model.init_weights()model.eval()# Dummy 3D volume: [B, C, H, W, D], every spatial dim a multiple of patch_size (16).dummy = torch.rand(1, 1, 48, 48, 48) * 2 - 1print("dummy input shape:", tuple(dummy.shape))with torch.no_grad():    out = model.forward_features(dummy)print("x_norm_clstoken   shape:", tuple(out["x_norm_clstoken"].shape))   # (1, embed_dim)print("x_norm_patchtokens shape:", tuple(out["x_norm_patchtokens"].shape))  # (1, N_patches, embed_dim)print("x_storage_tokens  shape:", tuple(out["x_storage_tokens"].shape))

A feature bag for downstream use stores the per-crop features as a `[N_patches, embed_dim]` `.npy` array. For the released model `embed_dim == 1024`.

## Part 2 — Feature bag → classification head → logitsReconstruct the gated-attention classifier and run it on a released `.npy` feature bag.

In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as Fimport numpy as npfrom pathlib import Pathclass GatedAttentionPooler(nn.Module):    def __init__(self, M, L):        super().__init__()        self.V = nn.Sequential(nn.LayerNorm(M, eps=1e-6), nn.Linear(M, L), nn.Tanh())        self.U = nn.Sequential(nn.LayerNorm(M, eps=1e-6), nn.Linear(M, L), nn.Sigmoid())        self.w = nn.Linear(L, 1)    def forward(self, H):        A = self.w(self.V(H) * self.U(H)).transpose(1, 2)        A = F.softmax(A, dim=2)        return torch.bmm(A, H).squeeze(1)class GateFusionClassifier(nn.Module):    def __init__(self, emb_dim, hidden_dim, n_cls, dropout):        super().__init__()        self.attn = GatedAttentionPooler(M=emb_dim, L=hidden_dim)        self.post = nn.Sequential(nn.Linear(emb_dim, emb_dim), nn.GELU(), nn.Dropout(dropout))        self.head = nn.Sequential(nn.Linear(emb_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, n_cls))    def forward(self, x):        z = self.attn(x)        z = self.post(z)        return self.head(z)def load_classifier(ckpt_path, device="cpu"):    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True)    state = ckpt["model"]    if "config" in ckpt:  # released best.pt format        arch = ckpt["config"]["architecture"]        hidden_dim = arch["hidden_dim"]        dropout = arch["dropout"]        n_cls = len(next(iter(ckpt["config"]["tasks"].values()))["label_columns"])    else:  # downstream/finetune.py checkpoint.pt format        hidden_dim = ckpt["hidden_dim"]        dropout = ckpt["dropout"]        n_cls = ckpt["n_cls"]    feat_dim = ckpt["feature_dim"]    model = GateFusionClassifier(feat_dim, hidden_dim, n_cls, dropout)    model.load_state_dict(state, strict=True)    model.eval().to(device)    print(f"loaded classifier: feature_dim={feat_dim} hidden_dim={hidden_dim} n_cls={n_cls} dropout={dropout}")    return model

Point the paths at the released checkpoint and demo features (see README).

In [ ]:
CKPT = "weights/classification_best.pt"     # downstream classification checkpoint (GitHub Release)FEATS_DIR = Path("data/demo/features")            # directory of <PatientID>.npy feature bagsLABELS_CSV = Path("data/demo/labels.csv")         # PatientID + one-hot class columnsmodel = load_classifier(CKPT, device="cpu")# Run one feature bag forward to logits.import pandas as pddf = pd.read_csv(LABELS_CSV)pid = str(df.iloc[0]["PatientID"])feats = np.load(FEATS_DIR / f"{pid}.npy").astype(np.float32)   # [N_patches, 1024]x = torch.from_numpy(feats)[None]                              # [1, N_patches, 1024]with torch.no_grad():    logits = model(x)print(f"case={pid}  feature_shape={feats.shape}  logits={logits.numpy()}")

Loop over all demo cases and print the logits (up to the classification head only).

In [ ]:
with torch.no_grad():    for _, row in df.iterrows():        pid = str(row["PatientID"])        feats = np.load(FEATS_DIR / f"{pid}.npy").astype(np.float32)        logits = model(torch.from_numpy(feats)[None])        print(f"{pid}: logits={logits.squeeze(0).numpy()}")